# TwHIN-BERT Climate Stance Experiment

This notebook repeats the finished SBERT-style experiment with frozen TwHIN-BERT tweet embeddings. The Kaggle column is named `sentiment`, but it is treated here as climate-change stance: `1` is believer/pro and `-1` is denier/anti. Neutral `0` and news `2` are excluded.

## Setup

Load the dependencies, set reproducibility constants, and locate the repository data file from either the repo root or the `notebooks/` directory.

In [1]:
from pathlib import Path
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")

SEED = 42
MODEL_NAME = "Twitter/twhin-bert-base"
BATCH_SIZE = 16
MAX_LENGTH = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Keep CPU runs predictable on laptops and shared machines.
if hasattr(torch, "set_num_threads"):
    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "twitter_sentiment_data.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/twitter_sentiment_data.csv")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_PATH = REPO_ROOT / "data" / "twitter_sentiment_data.csv"
OUTPUT_DIR = REPO_ROOT / "notebooks"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Repository root: {REPO_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

C:\Users\weipa\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository root: C:\Users\weipa\Documents\New project 3
Data path: C:\Users\weipa\Documents\New project 3\data\twitter_sentiment_data.csv
Device: cpu
Model: Twitter/twhin-bert-base


## Dataset

Filter the dataset to the binary stance task and verify that neutral/news rows are not included in `y_true`.

In [2]:
df = pd.read_csv(DATA_PATH)
label_counts = df["sentiment"].value_counts().sort_index()

stance_df = df[df["sentiment"].isin([-1, 1])].reset_index(drop=True)
stance_counts = stance_df["sentiment"].value_counts().sort_index()

assert df.shape == (43943, 3), f"Unexpected dataset shape: {df.shape}"
assert int(stance_counts.loc[-1]) == 3990, stance_counts.to_dict()
assert int(stance_counts.loc[1]) == 22962, stance_counts.to_dict()
assert len(stance_df) == 26952, len(stance_df)
assert set(stance_df["sentiment"].unique()) == {-1, 1}

texts = stance_df["message"].fillna("").astype(str).tolist()
y = stance_df["sentiment"].to_numpy()

label_summary = pd.DataFrame([
    {"Label": -1, "Stance": "denier/anti", "Rows": int(stance_counts.loc[-1])},
    {"Label": 1, "Stance": "believer/pro", "Rows": int(stance_counts.loc[1])},
])

print(f"Original dataset shape: {df.shape}")
print(f"Binary stance dataset shape: {stance_df.shape}")
print(f"Dropped neutral/news rows: {int((df['sentiment'].isin([0, 2])).sum())}")
display(label_summary)

Original dataset shape: (43943, 3)
Binary stance dataset shape: (26952, 3)
Dropped neutral/news rows: 16991


,Label,Stance,Rows
0,-1,denier/anti,3990
1,1,believer/pro,22962


## TwHIN-BERT Embeddings

Use TwHIN-BERT as a frozen encoder. Each tweet embedding is the attention-mask-aware mean of the final hidden states, then embeddings are L2-normalized before logistic regression.

In [3]:
def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def encode_texts_with_twhin(texts: list[str]) -> np.ndarray:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False).to(DEVICE)
    model.eval()

    batches = []
    total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
    for batch_number, start in enumerate(range(0, len(texts), BATCH_SIZE), start=1):
        batch_texts = texts[start:start + BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
        with torch.inference_mode():
            outputs = model(**inputs)
        pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
        batches.append(pooled.cpu().numpy().astype(np.float32))

        if batch_number % 250 == 0 or batch_number == total_batches:
            encoded = min(start + BATCH_SIZE, len(texts))
            print(f"Encoded {encoded:,} / {len(texts):,} tweets")

    return np.vstack(batches)

X_twhin = encode_texts_with_twhin(texts)
X_twhin = normalize(X_twhin)

print(f"TwHIN-BERT embedding matrix: {X_twhin.shape}")

Encoded 4,000 / 26,952 tweets
Encoded 8,000 / 26,952 tweets
Encoded 12,000 / 26,952 tweets
Encoded 16,000 / 26,952 tweets
Encoded 20,000 / 26,952 tweets
Encoded 24,000 / 26,952 tweets
Encoded 26,952 / 26,952 tweets
TwHIN-BERT embedding matrix: (26952, 768)


## Evaluation

Match the finished SBERT experiment: balanced logistic regression, 5-fold cross-validation predictions, and the same aggregate metrics.

In [4]:
classifier = LogisticRegression(max_iter=2000, class_weight="balanced")
y_pred = cross_val_predict(classifier, X_twhin, y, cv=5)

assert set(np.unique(y)) == {-1, 1}
assert set(np.unique(y_pred)).issubset({-1, 1})

metrics = {
    "Model": "TwHIN-BERT-base",
    "Task": "Climate stance (-1 denier/anti vs 1 believer/pro)",
    "Rows": len(y),
    "Macro-F1": f1_score(y, y_pred, average="macro", zero_division=0),
    "Weighted-F1": f1_score(y, y_pred, average="weighted", zero_division=0),
    "Macro-Precision": precision_score(y, y_pred, average="macro", zero_division=0),
    "Macro-Recall": recall_score(y, y_pred, average="macro", zero_division=0),
    "Accuracy": accuracy_score(y, y_pred),
}

aggregate_results = pd.DataFrame([metrics])
metric_cols = ["Macro-F1", "Weighted-F1", "Macro-Precision", "Macro-Recall", "Accuracy"]
aggregate_results[metric_cols] = aggregate_results[metric_cols].round(4)

report = classification_report(
    y,
    y_pred,
    labels=[-1, 1],
    target_names=["denier/anti (-1)", "believer/pro (1)"],
    output_dict=True,
    zero_division=0,
)
per_class_results = pd.DataFrame([
    {
        "Label": -1,
        "Stance": "denier/anti",
        "Precision": report["denier/anti (-1)"]["precision"],
        "Recall": report["denier/anti (-1)"]["recall"],
        "F1-score": report["denier/anti (-1)"]["f1-score"],
        "Support": int(report["denier/anti (-1)"]["support"]),
    },
    {
        "Label": 1,
        "Stance": "believer/pro",
        "Precision": report["believer/pro (1)"]["precision"],
        "Recall": report["believer/pro (1)"]["recall"],
        "F1-score": report["believer/pro (1)"]["f1-score"],
        "Support": int(report["believer/pro (1)"]["support"]),
    },
])
per_class_results[["Precision", "Recall", "F1-score"]] = per_class_results[["Precision", "Recall", "F1-score"]].round(4)

aggregate_results.to_csv(OUTPUT_DIR / "twhin_bert_stance_results.csv", index=False)
per_class_results.to_csv(OUTPUT_DIR / "twhin_bert_stance_class_report.csv", index=False)

print("TwHIN-BERT stance classification aggregate results")
display(aggregate_results)
print("TwHIN-BERT stance classification metrics by class")
display(per_class_results)
print("Metrics by class (classification_report format):")
print(classification_report(
    y,
    y_pred,
    labels=[-1, 1],
    target_names=["denier/anti (-1)", "believer/pro (1)"],
    digits=2,
    zero_division=0,
))

TwHIN-BERT stance classification aggregate results


,Model,Task,Rows,Macro-F1,Weighted-F1,Macro-Precision,Macro-Recall,Accuracy
0,TwHIN-BERT-base,Climate stance (-1 denier/anti vs 1 believer/pro),26952,0.6545,0.7867,0.6427,0.7413,0.7566


TwHIN-BERT stance classification metrics by class


,Label,Stance,Precision,Recall,F1-score,Support
0,-1,denier/anti,0.3454,0.7195,0.4667,3990
1,1,believer/pro,0.9400,0.7630,0.8423,22962


Metrics by class (classification_report format):
                  precision    recall  f1-score   support

denier/anti (-1)       0.35      0.72      0.47      3990
believer/pro (1)       0.94      0.76      0.84     22962

        accuracy                           0.76     26952
       macro avg       0.64      0.74      0.65     26952
    weighted avg       0.85      0.76      0.79     26952



## Notes

The reported task is stance classification, not sentiment analysis. The source column remains named `sentiment` only because that is the Kaggle CSV schema.